# Misha AI — Google Colab Test Bench

This notebook builds and tests the **`fix/background-voice-orb-and-commands` branch**. It clones the public GitHub repo, installs a Java 17 + Android SDK + Gradle build environment, runs the real Kotlin/JUnit unit tests, builds a debug APK, and optionally smoke-tests the configured public backend.

**Limits:** Colab cannot validate Android microphone capture, background microphone restrictions, floating-window permission, camera hardware, calls, or third-party app UI. Those require installing the APK on the phone. Do not upload API keys, signing keys, private contact data, or personal voice recordings to the public repository.

Run cells from top to bottom. A free CPU runtime is sufficient; GPU is not needed for Android build/tests.

In [ ]:
import os, pathlib, subprocess, textwrap
REPO_URL = "https://github.com/pritpatel872008-byte/-Misha-AI-Assistant.git"
BRANCH = "fix/background-voice-orb-and-commands"
WORKDIR = pathlib.Path("/content/Misha-AI-Assistant")
if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORKDIR)], check=True)
print("Repository:", WORKDIR)
print("Branch:", subprocess.check_output(["git","-C",str(WORKDIR),"branch","--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git","-C",str(WORKDIR),"rev-parse","--short","HEAD"], text=True).strip())
assert (WORKDIR / "android" / "app" / "build.gradle.kts").exists(), "Android Gradle project not found"

## 2. Install Android build tools

This installs OpenJDK 17, Android SDK platform 35/build tools 35.0.0, and Gradle 8.9 (the version required by this repository's Android Gradle Plugin). First run can take several minutes.

In [ ]:
import os, pathlib, subprocess, shutil, zipfile
subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk wget unzip", shell=True, check=True)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["ANDROID_SDK_ROOT"] = "/content/android-sdk"
os.environ["ANDROID_HOME"] = os.environ["ANDROID_SDK_ROOT"]
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

def download_verified(url, destination):
    destination = pathlib.Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    subprocess.run(["wget", "--tries=5", "--timeout=60", "--waitretry=3", "-q", url, "-O", str(destination)], check=True)
    with zipfile.ZipFile(destination) as archive:
        bad = archive.testzip()
        if bad:
            raise RuntimeError(f"Downloaded archive is corrupt at: {bad}")
    return destination

sdk_zip = download_verified(
    "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip",
    "/tmp/android-commandlinetools.zip"
)
sdk = pathlib.Path(os.environ["ANDROID_SDK_ROOT"])
latest = sdk / "cmdline-tools" / "latest"
shutil.rmtree(latest, ignore_errors=True)
shutil.rmtree("/tmp/android-cmdline", ignore_errors=True)
latest.mkdir(parents=True, exist_ok=True)
subprocess.run(["unzip", "-q", str(sdk_zip), "-d", "/tmp/android-cmdline"], check=True)
subprocess.run(["bash", "-lc", f"cp -r /tmp/android-cmdline/cmdline-tools/. '{latest}/'"], check=True)

sdkmanager = str(latest / "bin" / "sdkmanager")
os.environ["PATH"] = str(latest / "bin") + ":" + str(sdk / "platform-tools") + ":" + os.environ["PATH"]
subprocess.run(f"yes | '{sdkmanager}' --sdk_root='{sdk}' --licenses", shell=True, check=False)
subprocess.run([sdkmanager, "--sdk_root=" + str(sdk), "platform-tools", "platforms;android-35", "build-tools;35.0.0"], check=True)

gradle_zip = download_verified(
    "https://services.gradle.org/distributions/gradle-8.9-bin.zip",
    "/tmp/gradle-8.9-bin.zip"
)
shutil.rmtree("/opt/gradle-8.9", ignore_errors=True)
subprocess.run(["unzip", "-q", str(gradle_zip), "-d", "/opt"], check=True)
os.environ["PATH"] = "/opt/gradle-8.9/bin:" + os.environ["PATH"]

print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True))
print(subprocess.check_output(["gradle", "--version"], text=True).splitlines()[0:8])
print("Android SDK ready:", sdk)


## 3. Run the actual Android unit tests

This runs the repository's JUnit tests against the Kotlin source. A red result is a real failure to investigate, not a simulated pass.

In [ ]:
import subprocess, os
os.chdir(WORKDIR / "android")
result = subprocess.run(["gradle", ":app:testDebugUnitTest", "--no-daemon", "--stacktrace"], text=True)
print("Unit test exit code:", result.returncode)
assert result.returncode == 0, "Android unit tests failed — inspect the Gradle output above."

## 4. Build the debug APK

Only runs after the unit-test cell succeeds. This produces an installable debug APK, not a release-signed APK.

In [ ]:
import pathlib, subprocess
result = subprocess.run(["gradle", ":app:assembleDebug", "--no-daemon", "--stacktrace"], text=True)
apk = WORKDIR / "android/app/build/outputs/apk/debug/app-debug.apk"
print("Build exit code:", result.returncode)
print("APK exists:", apk.exists())
if apk.exists(): print("APK size:", round(apk.stat().st_size / (1024*1024), 2), "MB")
assert result.returncode == 0 and apk.exists(), "Debug APK build failed."

In [ ]:
from google.colab import files
files.download(str(apk))

## 5. Optional public-backend smoke test

This checks the health and search endpoints currently documented in the repository README. It does not test the Android app's full command routing. If the service is asleep, rate-limited, or unavailable, record that separately from Android build/test results.

In [ ]:
import requests
API_BASE = "https://misha-search-backend.vercel.app"
for path in ["/api/health", "/api/search?q=Android%20security"]:
    try:
        response = requests.get(API_BASE + path, timeout=25)
        print(path, "HTTP", response.status_code)
        print(response.text[:1200])
        response.raise_for_status()
    except Exception as exc:
        print(path, "FAILED:", repr(exc))


## 6. Optional: inspect the exported Hey Misha model ZIP

Upload the ZIP exported from the Heed notebook. This cell only verifies that the expected files are present and reads the ONNX model input/output metadata. It does **not** claim Android wake-word integration or live-microphone success. The ONNX model needs the exact Heed preprocessing pipeline before inference.

In [ ]:
from google.colab import files
import zipfile, pathlib, os
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert zip_names, "Upload the Heed export ZIP file."
model_zip = pathlib.Path("/content") / zip_names[0]
with zipfile.ZipFile(model_zip) as z:
    names = z.namelist()
    print("ZIP files:", names)
    required = ["wake.onnx", "wake.json"]
    for item in required:
        matches = [n for n in names if n.endswith(item)]
        print(item, "FOUND" if matches else "MISSING", matches[:2])
    assert all(any(n.endswith(item) for n in names) for item in required), "Required model/config file missing."
    extract_dir = pathlib.Path("/content/hey_misha_export")
    z.extractall(extract_dir)
!pip -q install onnxruntime
import onnxruntime as ort
onnx_path = next(extract_dir.rglob("wake.onnx"))
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print("ONNX inputs:", [(x.name, x.shape, x.type) for x in session.get_inputs()])
print("ONNX outputs:", [(x.name, x.shape, x.type) for x in session.get_outputs()])
print("Metadata inspection passed. Preprocessing/inference and Android integration remain separate tests.")

## 7. Real-phone acceptance checklist (must be done on Android)

After installing the APK, test each item manually and record the actual result:

- [ ] Say “Hey Misha” once → exactly one floating assistant popup appears.
- [ ] Popup opens the command-listening state and captures a spoken command.
- [ ] After one command, listening returns to idle without repeated popup/open/close loops.
- [ ] “Open Instagram” launches Instagram; app-internal search may require supported UI automation or a documented intent.
- [ ] “Take a photo” opens the camera; automatic shutter capture requires an explicit, permission-safe implementation.
- [ ] Torch on/off works.
- [ ] Calling resolves exact contacts and asks before placing a call.
- [ ] Messaging opens the selected app with the correct recipient/draft; the user reviews and sends.
- [ ] Screen-off/background behavior is tested with Android microphone foreground-service notification and battery settings.

Do not mark a phone-only item as passed based on Colab output.

## Results to share

Send the unit-test summary and APK build result first. If a cell fails, send the **first error block**, not just the last line. Keep the exported model ZIP private; do not commit your personal voice model or recordings to this public GitHub repository without deciding to publish them.